In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sys, os
sys.path.insert(0, '..')
from ssp_bayes_opt import sspspace
import tensorflow as tf
import sklearn
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import time



domain_dim=2
n_seeds = 10
n_test_pts = [1,100]
train_times = np.zeros(n_seeds)
methods = ['from-set', 'direct-optim', 'network', 'network-optim']
sample_decode_times = np.zeros((len(n_test_pts),4,n_seeds))
sample_decode_accur = np.zeros((len(n_test_pts),4,n_seeds))
for i in range(n_seeds):
    np.random.seed(i)
    tf.set_random_seed(i)
    ssp_space = sspspace.HexagonalSSPSpace(domain_dim,
                            n_rotates=6, 
                            n_scales=6, 
                            scale_min=2*np.pi/np.sqrt(6) - 0.5,
                            scale_max=2*np.pi/np.sqrt(6) + 0.5,
                            domain_bounds= np.tile([-1,1],(domain_dim,1)),
                            length_scale=0.1)
    start = time.time()
    history = ssp_space.train_decoder_net(n_training_pts=200000,n_hidden_units = 12,
                          learning_rate=1e-3,n_epochs = 20)
    end= time.time()
    train_times[i] = end - start

    sample_ssps, sample_points = ssp_space.get_sample_pts_and_ssps(method='grid', 
                        num_points_per_dim=300)
    
    for j in range(2):
        pt = np.random.rand((n_test_pts[j],domain_dim))*2 - 1
        ssp_pt = ssp_space.encode(pt)

        ##
        for method,k in enumerate(methods):
            start = time.time()
            decoded_pt = ssp_space.decode(method=method,samples = (sample_ssps, sample_points))
            end= time.time()
            sample_decode_times[j,k,i] = start - end
            sample_decode_accur[j,k,i] = np.mean((decoded_pt - pt)**2)





In [ ]:
def get_stats(data):
    (num_trials, _) = data.shape
    return np.mean(data, axis=0), np.std(data, axis=0) * 1.96 / np.sqrt(num_trials)

print("Avg training time: \n")
print(get_stats(train_times[:,None]))
for method,k in enumerate(methods):
    for n, j in enumerate(n_test_pts):
    print(f"Average {method} time ({n} sample): \n")
    print(get_stats(sample_decode_times[j,k,:][:,None]))
    
